# 03 — Within-Variant LLM Confidence

For each prompt variant, how much do the LLM services (OpenAI, Claude, Gemini, Ollama) agree with each other?

High agreement within a variant means that within a prompt, the LLMs converge on the same **exact translation** regardless of which model you ask.  
Low agreement means the models disagree, though that could be for capitalization or whitespace differences, as well as more substantive disagreements.

Baseline services (GT, EasyNMT, Lingvanex, Wikipedia) are prompt-invariant and shown as a reference, not as part of the LLM scoring pool.

Merges per-service files from `direct_services/` and `prompt_services/` at evaluation time — no combined staging files required.

> **Deprecated.** This notebook has been superseded by [`03_prompt_service_nexus.ipynb`](03_prompt_service_nexus.ipynb) (§2 — Within Prompt × Across Services), which integrates this analysis into a unified 2×2 framework. This file is retained for reference only and is no longer maintained.

In [1]:
import os
import sys
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

sys.path.insert(0, os.path.abspath('..'))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_within_variant import (
    run_confidence_evaluation,
    ALL_VARIANTS,
)

DATA_DIR     = get_data_directory_path()
TARGET_TERMS = ['Digital Humanities']
EVAL_DIR     = os.path.join(DATA_DIR, 'translated_terms', 'digital_humanities', 'evaluation')
VARIANTS     = ALL_VARIANTS

print(f'DATA_DIR: {DATA_DIR}')
print(f'Variants: {VARIANTS}')

Retrieving Coding DH data directory path...

DATA_DIR: /Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets
Variants: ['minimal', 'expert_persona', 'native_rationale', 'judge']


In [2]:
# Run scoring — writes confidence_scores.csv and confidence_summary.csv
scored_df, summary_df = run_confidence_evaluation(
    data_directory_path=DATA_DIR,
    target_terms=TARGET_TERMS,
    variants=VARIANTS,
    output_dir=EVAL_DIR,
)
print(f'\nScored rows:  {len(scored_df)}')
print(f'Summary rows: {len(summary_df)}')

📊 Scoring: Digital Humanities

  ✓ Loaded 858 rows for variant 'minimal'

  ✓ Loaded 858 rows for variant 'expert_persona'

  ✓ Loaded 858 rows for variant 'native_rationale'

  ✓ Loaded 858 rows for variant 'judge'

✓ Outputs written to: 
/Users/zleblanc/CodingDH/translation_transmogrification_pipeline/datasets/translated_terms/digital_humanities/evalu
ation

  confidence_scores.csv  : 3432 rows

  confidence_summary.csv : 4 rows

                              Confidence Scoring Summary (LLM agreement per variant)                               
┏━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ term       ┃ variant    ┃ total_lan… ┃ language… ┃ mean_llm_… ┃ median_l… ┃ std_llm_c… ┃ cv_llm_c… ┃ llm_uniqu… ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━┩
│ Digital    │ minimal    │ 858        │ 858       │ 0.3377     │ 0.25      │ 0.1652     │ 0.4892    │ 3.63       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ expert_pe… │ 858        │ 858       │ 0.3281     │ 0.25      │ 0.1614     │ 0.4919    │ 3.66       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ native_ra… │ 858        │ 858       │ 0.3363     │ 0.25      │ 0.1558     │ 0.4632    │ 3.55       │
│ Humanities │            │            │           │            │           │            │           │            │
│ Digital    │ judge      │ 858        │ 858       │ 0.4961     │ 0.5       │ 0.2434     │ 0.4906    │ 2.97       │
│ Humanities │            │            │           │            │           │            │           │            │
└────────────┴────────────┴────────────┴───────────┴────────────┴───────────┴────────────┴───────────┴────────────┘


Scored rows:  3432
Summary rows: 4


In [3]:
summary_df

,term,variant,total_languages,languages_with_llm,mean_llm_confidence,median_llm_confidence,std_llm_confidence,cv_llm_confidence,llm_unique_candidates_mean,languages_with_baseline,mean_baseline_confidence
0,Digital Humanities,minimal,858,858,0.3377,0.25,0.1652,0.4892,3.63,257,0.7662
1,Digital Humanities,expert_persona,858,858,0.3281,0.25,0.1614,0.4919,3.66,257,0.7662
2,Digital Humanities,native_rationale,858,858,0.3363,0.25,0.1558,0.4632,3.55,257,0.7662
3,Digital Humanities,judge,858,858,0.4961,0.50,0.2434,0.4906,2.97,257,0.7662


## LLM confidence distribution by variant

In [ ]:
plot_data = scored_df[scored_df['llm_total_services'] > 0].copy()
variant_order = VARIANTS

box = alt.Chart(plot_data).mark_boxplot(extent='min-max').encode(
    x=alt.X('prompt_variant:N', sort=variant_order, title=None),
    y=alt.Y('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]), title='LLM confidence'),
    color=alt.Color('prompt_variant:N', sort=variant_order, legend=None),
    tooltip=['prompt_variant:N', 'llm_confidence:Q'],
).properties(width=300, height=280, title='LLM confidence distribution per variant')

# Mean unique candidates per variant — higher = more divergence
cands = (
    plot_data.groupby('prompt_variant')['llm_unique_candidates']
    .mean()
    .reset_index()
    .rename(columns={'llm_unique_candidates': 'mean_unique_candidates'})
)
bar = alt.Chart(cands).mark_bar().encode(
    x=alt.X('prompt_variant:N', sort=variant_order, title=None),
    y=alt.Y('mean_unique_candidates:Q', title='mean unique candidates', scale=alt.Scale(domain=[0, 4])),
    color=alt.Color('prompt_variant:N', sort=variant_order, legend=None),
    tooltip=['prompt_variant:N', alt.Tooltip('mean_unique_candidates:Q', format='.2f')],
).properties(width=300, height=280, title='Mean unique candidates per variant (4 = all models differ)')

bar_text = bar.mark_text(dy=-8, fontSize=10).encode(
    text=alt.Text('mean_unique_candidates:Q', format='.2f'))

box | (bar + bar_text)

## Heatmap: LLM confidence by language family × variant

In [ ]:
scored_df['language_family'] = scored_df['language_code'].apply(get_language_family)
# Note: sign languages are included here — LLM confidence is a valid signal even for signed
# languages (the model still produces a translation); unlike word-count metrics, confidence
# does not assume text-based grammar so no exclusion is needed.

heatmap_data = (
    scored_df[scored_df['llm_total_services'] > 0]
    .groupby(['language_family', 'prompt_variant'])['llm_confidence']
    .mean()
    .reset_index()
    .rename(columns={'llm_confidence': 'mean_confidence'})
)

# Sort families by mean confidence across variants
family_order = (
    heatmap_data.groupby('language_family')['mean_confidence']
    .mean()
    .sort_values(ascending=False)
    .index.tolist()
)

rect = alt.Chart(heatmap_data).mark_rect().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('language_family:N', sort=family_order, title=None),
    color=alt.Color(
        'mean_confidence:Q',
        scale=alt.Scale(scheme='redyellowgreen', domain=[0, 1]),
        title='mean LLM confidence',
    ),
    tooltip=['language_family:N', 'prompt_variant:N', alt.Tooltip('mean_confidence:Q', format='.2f')],
)

text = rect.mark_text(fontSize=9).encode(
    text=alt.Text('mean_confidence:Q', format='.2f'),
    color=alt.condition(
        alt.datum.mean_confidence > 0.5,
        alt.value('black'),
        alt.value('white'),
    ),
)

(rect + text).properties(
    width=350, height=500,
    title='Mean LLM confidence by language family and prompt variant',
)

In [ ]:
# family_variant = (
#     scored_df[scored_df['llm_total_services'] > 0]
#     .groupby(['language_family', 'prompt_variant'])['llm_confidence']
#     .mean()
#     .reset_index()
#     .rename(columns={'llm_confidence': 'mean_llm_confidence'})
# )

# family_counts = (
#     scored_df[scored_df['llm_total_services'] > 0]
#     .groupby('language_family')['language_code']
#     .nunique()
#     .reset_index(name='n_languages')
# )

# family_variant = family_variant.merge(family_counts, on='language_family')
# family_variant['family_label'] = (
#     family_variant['language_family'] + ' (n=' + family_variant['n_languages'].astype(str) + ')'
# )

# selection = alt.selection_point(bind='legend', fields=['family_label'])
# alt.Chart(family_variant).mark_line(point=True).encode(
#     x=alt.X('prompt_variant:N', sort=VARIANTS, title='Prompt variant'),
#     y=alt.Y('mean_llm_confidence:Q', scale=alt.Scale(domain=[0, 1]), title='Mean LLM confidence'),
#     color=alt.Color('family_label:N', title='Language family', legend=alt.Legend(symbolLimit=0, labelLimit=0, columns=2), scale=alt.Scale(scheme='tableau20')),
#     tooltip=[
#         'family_label:N',
#         'prompt_variant:N',
#         alt.Tooltip('mean_llm_confidence:Q', format='.3f'),
#     ],
# 	opacity=alt.condition(selection, alt.value(1), alt.value(0.2)),
# ).add_params(selection).properties(
#     width=500, height=380,
#     title='Mean LLM Confidence per Variant by Language Family',
# )

alt.Chart(...)

## Most divergent rows (LLMs all disagree)

Languages where all four LLM services gave a different translation — the most interesting cases for the disagreement typology analysis.

In [7]:
display_cols = [
    'language_code', 'language_name', 'prompt_variant',
    'llm_confidence', 'llm_unique_candidates', 'llm_best_candidate',
    'llm_candidate_distribution',
]
most_divergent = (
    scored_df[scored_df['llm_total_services'] > 0]
    .sort_values('llm_confidence')
    .head(40)
)
most_divergent[display_cols]

,language_code,language_name,prompt_variant,llm_confidence,llm_unique_candidates,llm_best_candidate,llm_candidate_distribution
0,aa,Afar,minimal,0.25,4,Humaanitiyo Digitaala,"{'Humaanitiyo Digitaala': 1, 'Dijitaal Insinaa..."
1800,pis,Pijin,native_rationale,0.25,4,Digital Humanities,"{'Digital Humanities': 1, 'Dijital Humanitis':..."
1801,pko,Pökoot,native_rationale,0.25,4,Thorо́p Pорая і̱ро,"{'Thorо́p Pорая і̱ро': 1, 'Sapienta nē Teknolō..."
1803,pnt,Pontic,native_rationale,0.25,4,Ψηφιακός Ανθρωπιστικός,"{'Ψηφιακός Ανθρωπιστικός': 1, 'Ψηφιακαί Ανθρωπ..."
1804,pon,Pohnpeian,native_rationale,0.25,4,Humaniti Digital,"{'Humaniti Digital': 1, 'Padahk en Aramas Digi..."
1806,prd,Parsi-Dari,native_rationale,0.25,4,انسانیات دیجیتال,"{'انسانیات دیجیتال': 1, 'علوم انسانی دیجیتالی'..."
1807,prg,Prussian,native_rationale,0.25,4,Digitāle Humanistike,"{'Digitāle Humanistike': 1, 'Digitālai Sinnisk..."
1808,pro,Old Provençal,native_rationale,0.25,4,umanitats digitals,"{'umanitats digitals': 1, 'Artas Digitalas': 1..."
1809,puu,Punu,native_rationale,0.25,4,Imbône Ebe y'Ituta,"{""Imbône Ebe y'Ituta"": 1, 'Dibundu di Bikaniss..."
1810,ria,Riang (India),native_rationale,0.25,4,पाब्लाङ हिडोङमाथ,"{'पाब्लाङ हिडोङमाथ': 1, 'Digital Humanities': ..."


In [8]:
# Languages where all LLMs gave different translations (confidence = 0.25 with 4 services)
fully_divergent = (
    scored_df[
        (scored_df['llm_total_services'] >= 4) &
        (scored_df['llm_confidence'] == 0.25)
    ]
    .groupby('language_code')
    .size()
    .reset_index(name='variants_fully_divergent')
    .sort_values('variants_fully_divergent', ascending=False)
)

print(f'Languages fully divergent in at least 1 variant: {len(fully_divergent)}')
print(f'Fully divergent in all 4 variants: {(fully_divergent["variants_fully_divergent"] == 4).sum()}')

alt.Chart(fully_divergent.head(30)).mark_bar().encode(
    y=alt.Y('language_code:N', sort='-x', title=None),
    x=alt.X('variants_fully_divergent:Q', title='variants where all 4 LLMs disagreed',
            scale=alt.Scale(domain=[0, 4])),
    tooltip=['language_code:N', 'variants_fully_divergent:Q'],
).properties(width=350, height=500, title='Most consistently divergent languages (top 30)')

Languages fully divergent in at least 1 variant: 741
Fully divergent in all 4 variants: 240


alt.Chart(...)

## Baseline vs LLM confidence comparison

Direct services (GT/EasyNMT/Lingvanex/Wikipedia) are prompt-invariant, so their agreement should be the same across all variants. This is a sanity check.

In [9]:
comp = scored_df[scored_df['llm_total_services'] > 0].copy()
comp_mean = (
    comp.groupby('prompt_variant')[['llm_confidence', 'baseline_confidence']]
    .mean()
    .reindex(VARIANTS)
    .reset_index()
    .melt(id_vars='prompt_variant', var_name='source', value_name='mean_confidence')
)
comp_mean['source'] = comp_mean['source'].map({
    'llm_confidence': 'LLM (varies per variant)',
    'baseline_confidence': 'Baseline (invariant)',
})

alt.Chart(comp_mean).mark_bar().encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('mean_confidence:Q', title='Mean confidence', scale=alt.Scale(domain=[0, 1])),
    xOffset='source:N',
    color=alt.Color('source:N', title=None),
    tooltip=['prompt_variant:N', 'source:N', alt.Tooltip('mean_confidence:Q', format='.3f')],
).properties(width=400, height=280, title='LLM vs baseline service confidence per variant')

alt.Chart(...)

## Difference Type Analysis

Does the *kind* of disagreement predict score reliability? Each scored row carries an `llm_difference_types` field that records the pairwise difference categories across services:

| Type | Meaning |
|---|---|
| `all_identical` | All services returned the same string — score is 1.0 by definition |
| `capitalization` | Same text, different case only |
| `whitespace` | Same text, different surrounding whitespace |
| `both` | Capitalization + whitespace together, no content change |
| `content` | Genuine content difference |

Three questions to answer:
1. How common are trivial-only differences vs genuine content disagreements?
2. Do rows with only trivial differences cluster at different confidence levels than content disagreements?
3. How much would a weighted algorithm (treating capitalization/whitespace as the same candidate) actually move the scores — and is that movement worth the added complexity?

In [10]:
import ast

# Rows where at least 2 services contributed (single-service rows have no pairwise comparison)
diff_df = scored_df[scored_df['llm_total_services'] >= 2].copy()

# Classify each row's difference type into three tiers
def classify_diff_tier(d):
    if pd.isna(d) or str(d) in ('no_differences', 'all_identical', 'unknown', ''):
        return 'all_identical'
    types = set(str(d).split(','))
    if types <= {'capitalization', 'whitespace', 'both'}:
        return 'trivial_only'
    return 'has_content'

diff_df['diff_tier'] = diff_df['llm_difference_types'].apply(classify_diff_tier)

TIER_ORDER  = ['all_identical', 'trivial_only', 'has_content']
TIER_LABELS = {
    'all_identical': 'All identical (score = 1.0)',
    'trivial_only':  'Trivial only (capitalization / whitespace)',
    'has_content':   'Content differences',
}
TIER_COLOURS = {
    'all_identical': '#2ca02c',
    'trivial_only':  '#ff7f0e',
    'has_content':   '#d62728',
}

diff_df['tier_label'] = diff_df['diff_tier'].map(TIER_LABELS)

# Summary
tier_counts = diff_df['diff_tier'].value_counts().reset_index()
tier_counts.columns = ['diff_tier', 'n']
tier_counts['pct'] = (tier_counts['n'] / len(diff_df) * 100).round(1)
tier_counts['label'] = tier_counts['diff_tier'].map(TIER_LABELS)
tier_counts['pct_label'] = tier_counts.apply(lambda r: f"{r['n']} ({r['pct']}%)", axis=1)

print(f"Rows with ≥2 LLM services: {len(diff_df)}")
print()
for _, row in tier_counts.sort_values('diff_tier').iterrows():
    print(f"  {row['label']}: {row['pct_label']}")

# Raw frequency of each difference_types value (useful for seeing combos)
print("\nRaw difference_types values:")
raw_freq = diff_df['llm_difference_types'].fillna('unknown').value_counts()
print(raw_freq.to_string())

Rows with ≥2 LLM services: 3432

  All identical (score = 1.0): 103 (3.0%)
  Content differences: 3306 (96.3%)
  Trivial only (capitalization / whitespace): 23 (0.7%)

Raw difference_types values:
llm_difference_types
content                   3250
all_identical              103
capitalization,content      56
capitalization              23


In [ ]:
# Q1: Distribution of difference tiers
bar = alt.Chart(tier_counts).mark_bar().encode(
    x=alt.X('n:Q', title='Rows'),
    y=alt.Y('label:N', sort=[TIER_LABELS[t] for t in TIER_ORDER], title=None),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER, range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        legend=None),
    tooltip=['label:N', 'n:Q', 'pct:Q'],
)
bar_text = alt.Chart(tier_counts).mark_text(align='left', dx=4, fontSize=11).encode(
    x=alt.X('n:Q'),
    y=alt.Y('label:N', sort=[TIER_LABELS[t] for t in TIER_ORDER]),
    text='pct_label:N',
)

# Q2: Confidence distribution by tier (excluding all-identical rows — they're trivially 1.0)
non_identical = diff_df[diff_df['diff_tier'] != 'all_identical'].copy()
box = alt.Chart(non_identical).mark_boxplot(extent='min-max').encode(
    x=alt.X('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]), title='LLM confidence'),
    y=alt.Y('tier_label:N',
            sort=[TIER_LABELS['trivial_only'], TIER_LABELS['has_content']],
            title=None),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER, range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        legend=None),
    tooltip=['tier_label:N', alt.Tooltip('llm_confidence:Q', format='.3f')],
).properties(width=420, height=120,
             title='Confidence distribution by diff tier (excluding all-identical)')

# Mean confidence per tier × variant
tier_variant = (
    non_identical
    .groupby(['prompt_variant', 'diff_tier'])['llm_confidence']
    .mean()
    .reset_index()
)
tier_variant['tier_label'] = tier_variant['diff_tier'].map(TIER_LABELS)

line = alt.Chart(tier_variant).mark_line(point=True).encode(
    x=alt.X('prompt_variant:N', sort=VARIANTS, title=None),
    y=alt.Y('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]), title='Mean confidence'),
    color=alt.Color('tier_label:N',
        scale=alt.Scale(
            domain=[TIER_LABELS['trivial_only'], TIER_LABELS['has_content']],
            range=[TIER_COLOURS['trivial_only'], TIER_COLOURS['has_content']]
        ), title='Difference tier'),
    tooltip=['prompt_variant:N', 'tier_label:N', alt.Tooltip('llm_confidence:Q', format='.3f')],
).properties(width=380, height=200, title='Mean confidence by tier per variant')

(bar + bar_text).properties(width=380, height=140, title='Difference tier distribution') | box | line

In [ ]:
# Q3: Weighted algorithm simulation
# Parse llm_service_values and recompute confidence after normalizing (.strip().lower()).
# This is what the weighted algorithm would produce. Compare the delta to the exact-match score.

from collections import Counter as _Counter

def _sim_normalized_confidence(service_values_str: str) -> float:
    try:
        sv = ast.literal_eval(str(service_values_str))
    except Exception:
        return float('nan')
    valid = [
        str(v).strip().lower() for v in sv.values()
        if v and str(v) not in ('nan', 'None', '') and pd.notna(v)
    ]
    if not valid:
        return float('nan')
    best_count = _Counter(valid).most_common(1)[0][1]
    return round(best_count / len(valid), 4)

sim_df = diff_df.copy()
sim_df['sim_confidence'] = sim_df['llm_service_values'].apply(_sim_normalized_confidence)
sim_df['confidence_delta'] = (sim_df['sim_confidence'] - sim_df['llm_confidence']).round(4)

changed = sim_df[sim_df['confidence_delta'] > 0]
print(f"Rows where normalised score > exact-match score: {len(changed)} / {len(sim_df)} ({len(changed)/len(sim_df)*100:.1f}%)")
print(f"\nAmong rows that change:")
print(f"  Mean delta:   +{changed['confidence_delta'].mean():.4f}")
print(f"  Median delta: +{changed['confidence_delta'].median():.4f}")
print(f"  Max delta:    +{changed['confidence_delta'].max():.4f}")
print(f"\nDiff tier of changed rows:")
print(changed['diff_tier'].value_counts().to_string())

# Delta distribution chart
delta_plot = changed.copy()
delta_plot['tier_label'] = delta_plot['diff_tier'].map(TIER_LABELS)

hist = alt.Chart(delta_plot).mark_bar(opacity=0.85).encode(
    x=alt.X('confidence_delta:Q',
            bin=alt.Bin(step=0.0834),  # 1/12 — smallest meaningful step with 4 services
            title='Score increase under normalised algorithm'),
    y=alt.Y('count():Q', title='Rows'),
    color=alt.Color('tier_label:N',
        scale=alt.Scale(
            domain=[TIER_LABELS['trivial_only'], TIER_LABELS['has_content']],
            range=[TIER_COLOURS['trivial_only'], TIER_COLOURS['has_content']]
        ), title='Diff tier'),
    tooltip=['tier_label:N', 'count():Q'],
).properties(
    width=380, height=220,
    title=alt.TitleParams(
        'Score delta: normalised vs exact-match (rows that change only)',
        subtitle='If trivial_only rows dominate here, the weighted algorithm adds real signal.',
        subtitleFontSize=10, subtitleColor='#666',
    )
)

# Scatter: exact-match vs simulated — points above diagonal changed
diag_data = pd.DataFrame({'x': [0, 1], 'y': [0, 1]})
diag = alt.Chart(diag_data).mark_line(color='grey', strokeDash=[4, 4], opacity=0.5).encode(
    x='x:Q', y='y:Q'
)
scatter = alt.Chart(sim_df.dropna(subset=['sim_confidence'])).mark_circle(
    opacity=0.35, size=35
).encode(
    x=alt.X('llm_confidence:Q', scale=alt.Scale(domain=[0, 1]), title='Exact-match confidence'),
    y=alt.Y('sim_confidence:Q',  scale=alt.Scale(domain=[0, 1]), title='Normalised confidence'),
    color=alt.Color('diff_tier:N',
        scale=alt.Scale(domain=TIER_ORDER, range=[TIER_COLOURS[t] for t in TIER_ORDER]),
        title='Diff tier'),
    tooltip=[
        alt.Tooltip('llm_confidence:Q', format='.3f', title='exact-match'),
        alt.Tooltip('sim_confidence:Q',  format='.3f', title='normalised'),
        'diff_tier:N', 'prompt_variant:N',
    ],
).properties(width=280, height=280, title='Exact-match vs normalised confidence')

hist | (scatter + diag)

### Findings: weighted algorithm not warranted

Running against the Digital Humanities dataset (858 languages × 4 variants = 3,432 rows with ≥2 LLM services):

| Diff tier | Rows | % |
|---|---|---|
| All identical (score = 1.0) | 103 | 3.0% |
| **Content differences** | **3,306** | **96.3%** |
| Trivial only (capitalization / whitespace) | 23 | 0.7% |

Only **2.2% of rows** would change score under a normalised (case/whitespace-insensitive) algorithm. Of the 76 rows that would change, 53 still have genuine content differences — only 23 are pure capitalization/whitespace artefacts.

**Decision: keep exact-match scoring.** The 96.3% content-difference rate means the current strict metric is accurately reflecting real disagreement, not noise. The `difference_types` metadata column is sufficient for any row-level interpretation — no separate weighted algorithm is needed. The 23 trivial-only rows (~0.7%) are a known, small artefact that can be noted in analysis but do not justify a more complex scoring approach.